# Cross-Response Hallucination Scoring

This notebook provides a reproducible, offline-safe implementation of the
cross-response consistency analysis used for the DeepGenomeAgent evaluation.
It compares three independently generated responses for each gene, records
ordered entailment judgments, and aggregates contradiction evidence.

## Scope and Limitations

The workflow measures disagreement among repeated responses. It does not
establish biological truth and cannot detect a claim that is repeated
consistently across all responses. Entailment labels depend on a private
judge model when live evaluation is enabled. The public, default execution
performs input checks, pure-function checks, and optional aggregation of
existing logs without contacting any service.

## Method

Each response is split into overlapping windows of three sentences with a
two-sentence stride. Every window from response *i* is judged against the
full text of response *j* for all ordered pairs where *i* differs from *j*.
An entailment edge requires support of at least 0.6 and contradiction of at
most 0.2. Mutual edges define connected equivalence clusters. Normalized
semantic entropy is retained as a disagreement diagnostic, while the
reported contradiction summary is the mean window-level contradiction
evidence recovered from validated logs.

## Reproducibility Configuration

Private inputs and judge credentials are accepted only through environment
variables. Set `DEEPGENOME_RUN_LIVE_JUDGING=1` explicitly to permit API
requests; leaving it unset keeps execution offline.

In [ ]:
import os
from pathlib import Path


def optional_path(name: str) -> Path | None:
    value = os.getenv(name)
    return Path(value).expanduser().resolve() if value else None


QUERY_WORKBOOK = optional_path("DEEPGENOME_QUERY_WORKBOOK")
RESPONSE_ROOT = optional_path("DEEPGENOME_RESPONSE_ROOT")
JUDGMENT_DIR = optional_path("DEEPGENOME_JUDGMENT_DIR")
API_BASE_URL = os.getenv("DEEPGENOME_API_BASE_URL")
API_KEY = os.getenv("DEEPGENOME_API_KEY")
JUDGE_MODEL = os.getenv("DEEPGENOME_JUDGE_MODEL")
RUN_LIVE_JUDGING = os.getenv("DEEPGENOME_RUN_LIVE_JUDGING", "0") == "1"
MAX_CONCURRENT = 32


def find_repository_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (
            (candidate / "pyproject.toml").is_file()
            and (candidate / "reproduce.manifest.yaml").is_file()
        ):
            return candidate
    raise RuntimeError("Could not locate the repository root.")


REPOSITORY_ROOT = find_repository_root(Path.cwd())
GENE_LIST_PATH = (
    REPOSITORY_ROOT
    / "Supplementary Fig. 10-13"
    / "PhytoBench-Gene-for_plot"
    / "score.uncharacterized.tsv"
)

## Load and Validate Inputs

The public gene list is repository-backed. The query workbook and response
collections are private evaluation inputs and are validated only when their
environment-backed locations are supplied.

In [ ]:
import csv

import pandas as pd


MODEL_IDS = ("phytomni", "gemini", "openai", "grok")
RESPONSE_LAYOUTS = {
    "phytomni": (
        "phytomni-gene_function-md/{gene_id}_result.md",
        "phytomni-gene_function-md2/{gene_id}_result.md",
        "phytomni-gene_function-md3/{gene_id}_result.md",
    ),
    "gemini": (
        "Gemini-Un/Gemini-{gene_id}-1.md",
        "Gemini-Un/Gemini-{gene_id}-2.md",
        "Gemini-Un/Gemini-{gene_id}-3.md",
    ),
    "openai": (
        "OpenAI-Un/Openai-{gene_id}.md",
        "OpenAI-Un/{gene_id}_Rep2.md",
        "OpenAI-Un/{gene_id}_Rep3.md",
    ),
    "grok": (
        "Grok-Un/Grok-{gene_id}.md",
        "Grok-Un/Grok-{gene_id}-2.md",
        "Grok-Un/Grok-{gene_id}-3.md",
    ),
}


def load_gene_ids(path: Path) -> list[str]:
    if not path.is_file():
        raise FileNotFoundError(f"Gene list not found: {path}")
    with path.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle, delimiter="	")
        if not reader.fieldnames or "Gene" not in reader.fieldnames:
            raise ValueError("Gene list TSV must contain a 'Gene' column.")
        gene_ids = sorted(
            {
                row["Gene"].strip()
                for row in reader
                if row.get("Gene") and row["Gene"].strip()
            }
        )
    if not gene_ids:
        raise ValueError("Gene list TSV contains no gene IDs.")
    return gene_ids


def load_gene_queries(path: Path, expected_gene_ids: list[str]) -> dict[str, str]:
    if not path.is_file():
        raise FileNotFoundError(f"Query workbook not found: {path}")
    frame = pd.read_excel(path)
    required_columns = {"Gene ID", "Query"}
    missing_columns = sorted(required_columns.difference(frame.columns))
    if missing_columns:
        raise ValueError(
            f"Query workbook is missing columns: {', '.join(missing_columns)}"
        )
    query_rows = frame.loc[:, ["Gene ID", "Query"]].dropna()
    queries = {
        str(row["Gene ID"]).strip(): str(row["Query"]).strip()
        for _, row in query_rows.iterrows()
        if str(row["Gene ID"]).strip() and str(row["Query"]).strip()
    }
    missing_queries = [gene_id for gene_id in expected_gene_ids if gene_id not in queries]
    if missing_queries:
        preview = ", ".join(missing_queries[:5])
        raise ValueError(f"Query workbook lacks {len(missing_queries)} genes: {preview}")
    return queries


def response_paths(response_root: Path, model_id: str, gene_id: str) -> list[Path]:
    if model_id not in RESPONSE_LAYOUTS:
        raise ValueError(f"Unsupported model ID: {model_id}")
    return [
        response_root / template.format(gene_id=gene_id)
        for template in RESPONSE_LAYOUTS[model_id]
    ]


def validate_response_files(
    response_root: Path,
    expected_gene_ids: list[str],
) -> dict[str, list[str]]:
    if not response_root.is_dir():
        raise NotADirectoryError(f"Response root not found: {response_root}")
    missing_by_model: dict[str, list[str]] = {}
    for model_id in MODEL_IDS:
        missing = [
            gene_id
            for gene_id in expected_gene_ids
            if not all(
                path.is_file()
                for path in response_paths(response_root, model_id, gene_id)
            )
        ]
        missing_by_model[model_id] = missing
    return missing_by_model


def load_response_triplet(
    response_root: Path,
    model_id: str,
    gene_id: str,
) -> list[str]:
    paths = response_paths(response_root, model_id, gene_id)
    missing = [str(path) for path in paths if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"Missing response files: {missing}")
    responses = [path.read_text(encoding="utf-8") for path in paths]
    if any(not response.strip() for response in responses):
        raise ValueError(f"Empty response found for {model_id}/{gene_id}.")
    return responses


gene_ids = load_gene_ids(GENE_LIST_PATH)
print(f"Loaded {len(gene_ids)} unique genes from {GENE_LIST_PATH.relative_to(REPOSITORY_ROOT)}.")

## Pairwise Entailment Judgments

The following tagged cell is intentionally self-contained so its pure
calculations and log aggregation can be tested without executing private
I/O or a judge client.

In [ ]:
from math import log
from pathlib import Path
from typing import Any
import json

import networkx as nx
import numpy as np


WINDOW_SIZE_SENTENCES = 3
WINDOW_STRIDE_SENTENCES = 2
SUPPORT_THRESHOLD = 0.6
CONTRADICTION_THRESHOLD = 0.2
SEMANTIC_HIGH_RISK_THRESHOLD = 0.6
GENE_HIGH_CONTRADICTION_THRESHOLD = 0.6


def parse_judge_label(raw_response: str) -> str:
    answer = raw_response.strip().lower()
    if "entailment" in answer:
        return "entailment"
    if "contradiction" in answer:
        return "contradiction"
    return "neutral"


def edge_from_ratios(
    support_ratio: float,
    contradiction_ratio: float,
) -> bool:
    return (
        support_ratio >= SUPPORT_THRESHOLD
        and contradiction_ratio <= CONTRADICTION_THRESHOLD
    )


def equivalence_clusters(entailment_matrix: np.ndarray) -> list[list[int]]:
    graph = nx.Graph()
    response_count = entailment_matrix.shape[0]
    graph.add_nodes_from(range(response_count))
    for source in range(response_count):
        for target in range(source + 1, response_count):
            if entailment_matrix[source, target] and entailment_matrix[target, source]:
                graph.add_edge(source, target)
    return [sorted(component) for component in nx.connected_components(graph)]


def normalized_semantic_entropy(
    clusters: list[list[int]],
    response_count: int,
) -> float:
    if response_count <= 1:
        return 0.0
    entropy = 0.0
    for cluster in clusters:
        proportion = len(cluster) / response_count
        if proportion > 0:
            entropy -= proportion * log(proportion)
    normalized = entropy / log(response_count)
    if abs(normalized - 1.0) < 1e-12:
        return 1.0
    return min(1.0, normalized)


def semantic_entropy_is_high_risk(semantic_entropy: float) -> bool:
    return semantic_entropy > SEMANTIC_HIGH_RISK_THRESHOLD


def validate_judgment_records(
    records: list[dict[str, Any]],
    expected_response_count: int = 3,
) -> list[str]:
    errors: list[str] = []
    if not isinstance(records, list):
        return ["Judgment log must be a JSON list."]
    if any("error" in record for record in records if isinstance(record, dict)):
        errors.append("Judgment log contains API errors.")
    metadata = next(
        (record for record in records if record.get("type") == "metadata"),
        None,
    )
    if metadata is not None:
        response_ids = metadata.get("response_ids")
        if not isinstance(response_ids, list) or len(response_ids) != expected_response_count:
            errors.append("Metadata must contain exactly three response IDs.")
    summaries = [
        record
        for record in records
        if isinstance(record, dict) and record.get("type") == "version_pair_summary"
    ]
    expected_pairs = expected_response_count * (expected_response_count - 1)
    if summaries and len(summaries) != expected_pairs:
        errors.append(f"Expected {expected_pairs} pair summaries, found {len(summaries)}.")
    if not summaries and not any(
        "label" in record for record in records if isinstance(record, dict)
    ):
        errors.append("Judgment log contains neither pair summaries nor window labels.")
    return errors


def extract_gene_contradiction(
    records: list[dict[str, Any]],
) -> tuple[float, int, str] | None:
    summaries = [
        record
        for record in records
        if isinstance(record, dict) and record.get("type") == "version_pair_summary"
    ]
    if summaries:
        ratios = [float(record.get("contra_ratio", 0.0)) for record in summaries]
        if ratios:
            return sum(ratios) / len(ratios), len(ratios), "version_pair_summary"
    labels = [
        record.get("label")
        for record in records
        if isinstance(record, dict) and "label" in record
    ]
    if labels:
        contradictions = sum(
            isinstance(label, str) and label.lower().startswith("contra")
            for label in labels
        )
        return contradictions / len(labels), len(labels), "window_labels"
    return None


def aggregate_model_logs(
    model_id: str,
    gene_ids: list[str],
    judgment_dir: Path,
    threshold: float = 0.6,
    weighted: bool = False,
) -> dict[str, object]:
    gene_scores: list[dict[str, object]] = []
    missing_gene_ids: list[str] = []
    invalid_gene_ids: list[str] = []
    for gene_id in gene_ids:
        log_path = (
            judgment_dir
            / f"{model_id}__{gene_id}__rep_1-rep_2-rep_3.json"
        )
        if not log_path.is_file():
            missing_gene_ids.append(gene_id)
            continue
        try:
            records = json.loads(log_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError, UnicodeError):
            invalid_gene_ids.append(gene_id)
            continue
        try:
            validation_errors = validate_judgment_records(records)
            extracted = (
                extract_gene_contradiction(records)
                if not validation_errors
                else None
            )
        except (AttributeError, TypeError, ValueError):
            extracted = None
        if extracted is None:
            invalid_gene_ids.append(gene_id)
            continue
        ratio, evidence_count, source = extracted
        gene_scores.append(
            {
                "gene_id": gene_id,
                "contradiction_ratio": ratio,
                "evidence_count": evidence_count,
                "source": source,
                "high_contradiction": ratio >= threshold,
            }
        )

    valid_log_count = len(gene_scores)
    high_contradiction_count = sum(
        bool(score["high_contradiction"]) for score in gene_scores
    )
    if not gene_scores:
        average_ratio = None
    elif weighted:
        total_weight = sum(int(score["evidence_count"]) for score in gene_scores)
        average_ratio = (
            sum(
                float(score["contradiction_ratio"])
                * int(score["evidence_count"])
                for score in gene_scores
            )
            / total_weight
            if total_weight
            else 0.0
        )
    else:
        average_ratio = sum(
            float(score["contradiction_ratio"]) for score in gene_scores
        ) / valid_log_count

    high_rate = (
        high_contradiction_count / valid_log_count
        if valid_log_count
        else None
    )
    return {
        "model_id": model_id,
        "threshold": threshold,
        "weighted": weighted,
        "requested_gene_count": len(gene_ids),
        "valid_log_count": valid_log_count,
        "missing_log_count": len(missing_gene_ids),
        "invalid_log_count": len(invalid_gene_ids),
        "high_contradiction_count": high_contradiction_count,
        "high_contradiction_rate": high_rate,
        "average_contradiction_ratio": average_ratio,
        "mean_contradiction_ratio": average_ratio,
        "gene_scores": gene_scores,
        "missing_gene_ids": missing_gene_ids,
        "invalid_gene_ids": invalid_gene_ids,
    }

In [ ]:
import asyncio
import aiofiles
import nltk
from openai import AsyncOpenAI


class AsyncLongTextHallucinationDetector:
    def __init__(
        self,
        base_url: str,
        openai_api_key: str,
        model: str,
        max_concurrent: int = 32,
    ) -> None:
        self.model = model
        self.client = AsyncOpenAI(base_url=base_url, api_key=openai_api_key)
        self.semaphore = asyncio.Semaphore(max_concurrent)
        self.judgments_log: list[dict[str, object]] = []

    @staticmethod
    def split_into_windows(text: str) -> list[str]:
        sentences = nltk.sent_tokenize(text)
        if len(sentences) <= WINDOW_SIZE_SENTENCES:
            return [text]
        return [
            " ".join(sentences[start : start + WINDOW_SIZE_SENTENCES])
            for start in range(0, len(sentences), WINDOW_STRIDE_SENTENCES)
            if sentences[start : start + WINDOW_SIZE_SENTENCES]
        ]

    async def judge_entailment(
        self,
        hypothesis_window: str,
        context_text: str,
        source_index: int,
        target_index: int,
        window_index: int,
    ) -> str:
        prompt = f'''You are an expert entailment judge for long texts.

Context (full text):
{context_text}

Hypothesis (text segment to verify):
{hypothesis_window}

Question: Based on the Context, does the Hypothesis semantically follow or contradict?

Instructions:
- Answer 'entailment' if the Hypothesis is fully supported by the Context
- Answer 'contradiction' if the Hypothesis contradicts the Context  
- Answer 'neutral' if the Hypothesis is neither fully supported nor contradicted

Answer with exactly one word: entailment/contradiction/neutral'''
        record: dict[str, object] = {
            "type": "window_judgment",
            "source_index": source_index,
            "target_index": target_index,
            "window_index": window_index,
            "version_pair": [source_index, target_index],
            "hypothesis": hypothesis_window,
        }
        async with self.semaphore:
            try:
                response = await self.client.chat.completions.create(
                    model=self.model,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0,
                    max_tokens=10,
                )
                raw_response = response.choices[0].message.content or ""
                label = parse_judge_label(raw_response)
                record.update(
                    {"raw_response": raw_response.strip(), "label": label}
                )
            except Exception as error:
                label = "neutral"
                record.update({"error": str(error), "label": label})
        self.judgments_log.append(record)
        return label

    async def compute_version_entailment(
        self,
        windows: list[str],
        context_text: str,
        source_index: int,
        target_index: int,
    ) -> tuple[float, float]:
        labels = await asyncio.gather(
            *[
                self.judge_entailment(
                    window,
                    context_text,
                    source_index,
                    target_index,
                    window_index,
                )
                for window_index, window in enumerate(windows)
            ]
        )
        total = max(1, len(labels))
        support_ratio = labels.count("entailment") / total
        contradiction_ratio = labels.count("contradiction") / total
        return support_ratio, contradiction_ratio

    async def build_entailment_matrix(
        self,
        texts: list[str],
        windows_by_response: list[list[str]],
    ) -> np.ndarray:
        response_count = len(texts)
        matrix = np.zeros((response_count, response_count), dtype=bool)
        pairs = [
            (source_index, target_index)
            for source_index in range(response_count)
            for target_index in range(response_count)
            if source_index != target_index
        ]
        results = await asyncio.gather(
            *[
                self.compute_version_entailment(
                    windows_by_response[source_index],
                    texts[target_index],
                    source_index,
                    target_index,
                )
                for source_index, target_index in pairs
            ]
        )
        for (source_index, target_index), (
            support_ratio,
            contradiction_ratio,
        ) in zip(pairs, results, strict=True):
            has_edge = edge_from_ratios(
                support_ratio,
                contradiction_ratio,
            )
            matrix[source_index, target_index] = has_edge
            self.judgments_log.append(
                {
                    "type": "version_pair_summary",
                    "version_pair": [source_index, target_index],
                    "support_ratio": support_ratio,
                    "contra_ratio": contradiction_ratio,
                    "entailment_established": has_edge,
                }
            )
        return matrix

    async def analyze_responses(
        self,
        responses: list[str],
        model_id: str,
        gene_id: str,
        question: str,
        response_ids: list[str],
    ) -> dict[str, object]:
        if len(responses) != len(response_ids):
            raise ValueError("Responses and response IDs must have equal length.")
        if len(responses) < 2:
            raise ValueError("At least two responses are required.")
        self.judgments_log = [
            {
                "type": "metadata",
                "model_id": model_id,
                "gene_id": gene_id,
                "question": question,
                "response_ids": response_ids,
            }
        ]
        windows = [self.split_into_windows(response) for response in responses]
        matrix = await self.build_entailment_matrix(responses, windows)
        clusters = equivalence_clusters(matrix)
        semantic_entropy = normalized_semantic_entropy(
            clusters,
            len(responses),
        )
        return {
            "model_id": model_id,
            "gene_id": gene_id,
            "response_ids": response_ids,
            "semantic_entropy": semantic_entropy,
            "high_semantic_risk": semantic_entropy_is_high_risk(
                semantic_entropy
            ),
            "clusters": clusters,
            "entailment_matrix": matrix.tolist(),
        }

    def ordered_judgment_records(self) -> list[dict[str, object]]:
        metadata = [
            record
            for record in self.judgments_log
            if record.get("type") == "metadata"
        ]
        windows = sorted(
            (
                record
                for record in self.judgments_log
                if record.get("type") == "window_judgment"
            ),
            key=lambda record: (
                int(record["source_index"]),
                int(record["target_index"]),
                int(record["window_index"]),
            ),
        )
        summaries = sorted(
            (
                record
                for record in self.judgments_log
                if record.get("type") == "version_pair_summary"
            ),
            key=lambda record: tuple(record["version_pair"]),
        )
        return [*metadata, *windows, *summaries]

    async def save_judgments_to_file(self, path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        payload = json.dumps(
            self.ordered_judgment_records(),
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )
        async with aiofiles.open(path, "w", encoding="utf-8") as handle:
            await handle.write(payload + "\n")

## Semantic-Entropy Diagnostic

Entropy is zero when all responses occupy one equivalence cluster and one
when every response occupies its own cluster. With three responses, a
two-plus-one split has normalized entropy 0.5793801642856949. This diagnostic
is reported separately from the contradiction-log aggregate.

In [ ]:
assert normalized_semantic_entropy([[0, 1, 2]], 3) == 0.0
assert normalized_semantic_entropy([[0], [1], [2]], 3) == 1.0
assert np.isclose(
    normalized_semantic_entropy([[0, 1], [2]], 3),
    0.5793801642856949,
)
assert edge_from_ratios(0.6, 0.2)
assert not edge_from_ratios(np.nextafter(0.6, 0.0), 0.2)
assert not edge_from_ratios(0.6, np.nextafter(0.2, 1.0))
assert not semantic_entropy_is_high_risk(0.6)
assert semantic_entropy_is_high_risk(np.nextafter(0.6, 1.0))
print("Offline semantic-entropy checks passed.")

## Aggregate Contradiction Scores

Logs use the fixed name
`{model_id}__{gene_id}__rep_1-rep_2-rep_3.json`. Invalid, incomplete,
or API-error logs are inventoried but excluded. The unweighted mean gives
each gene equal influence; the optional weighted mean uses the number of
pair summaries or window labels as evidence weight. A gene is counted as
high contradiction when its ratio is at least 0.6.

In [ ]:
def aggregate_configured_model_logs(
    model_ids: tuple[str, ...],
    configured_gene_ids: list[str],
    judgment_dir: Path | None,
    threshold: float = 0.6,
    weighted: bool = False,
) -> dict[str, dict[str, object]]:
    if judgment_dir is None or not judgment_dir.is_dir():
        return {}
    return {
        model_id: aggregate_model_logs(
            model_id,
            configured_gene_ids,
            judgment_dir,
            threshold=threshold,
            weighted=weighted,
        )
        for model_id in model_ids
    }


model_log_summaries: dict[str, dict[str, object]] = {}

## Reproducibility Checks

Live judging requires all private paths and API settings, an explicit opt-in,
and a locally installed NLTK sentence-tokenizer resource. The notebook never
downloads language data or makes an implicit network request.

In [ ]:
async def run_live_evaluation() -> list[dict[str, object]]:
    if QUERY_WORKBOOK is None or RESPONSE_ROOT is None or JUDGMENT_DIR is None:
        raise RuntimeError("Private path configuration is incomplete.")
    if not API_BASE_URL or not API_KEY or not JUDGE_MODEL:
        raise RuntimeError("Judge API configuration is incomplete.")
    try:
        nltk.data.find("tokenizers/punkt_tab")
    except LookupError as error:
        raise RuntimeError(
            "NLTK punkt_tab is required for live judging; install it before execution."
        ) from error

    queries = load_gene_queries(QUERY_WORKBOOK, gene_ids)
    missing_by_model = validate_response_files(RESPONSE_ROOT, gene_ids)
    incomplete = {
        model_id: missing
        for model_id, missing in missing_by_model.items()
        if missing
    }
    if incomplete:
        counts = {
            model_id: len(missing) for model_id, missing in incomplete.items()
        }
        raise FileNotFoundError(f"Incomplete response triplets: {counts}")

    detector = AsyncLongTextHallucinationDetector(
        base_url=API_BASE_URL,
        openai_api_key=API_KEY,
        model=JUDGE_MODEL,
        max_concurrent=MAX_CONCURRENT,
    )
    live_results: list[dict[str, object]] = []
    for model_id in MODEL_IDS:
        for gene_id in gene_ids:
            result = await detector.analyze_responses(
                responses=load_response_triplet(
                    RESPONSE_ROOT,
                    model_id,
                    gene_id,
                ),
                model_id=model_id,
                gene_id=gene_id,
                question=queries[gene_id],
                response_ids=["rep_1", "rep_2", "rep_3"],
            )
            log_path = (
                JUDGMENT_DIR
                / f"{model_id}__{gene_id}__rep_1-rep_2-rep_3.json"
            )
            await detector.save_judgments_to_file(log_path)
            live_results.append(result)
    return live_results


assert GENE_LIST_PATH.is_file()
assert len(gene_ids) == len(set(gene_ids))
assert parse_judge_label("ENTAILMENT") == "entailment"
assert parse_judge_label("contradiction") == "contradiction"
assert parse_judge_label("uncertain") == "neutral"
inline_summaries = [
    {
        "type": "version_pair_summary",
        "version_pair": [index, index + 1],
        "contra_ratio": value,
    }
    for index, value in enumerate([0.1, 0.2, 0.3])
]
inline_summary_result = extract_gene_contradiction(inline_summaries)
assert inline_summary_result is not None
assert np.isclose(inline_summary_result[0], 0.2)
assert inline_summary_result[1:] == (3, "version_pair_summary")
inline_labels = [
    {"label": "entailment"},
    {"label": "contradiction"},
    {"label": "neutral"},
    {"label": "contradiction"},
]
assert extract_gene_contradiction(inline_labels) == (
    0.5,
    4,
    "window_labels",
)
assert validate_judgment_records([{"error": "timeout", "label": "neutral"}])

required_live_configuration = {
    "DEEPGENOME_QUERY_WORKBOOK": QUERY_WORKBOOK,
    "DEEPGENOME_RESPONSE_ROOT": RESPONSE_ROOT,
    "DEEPGENOME_JUDGMENT_DIR": JUDGMENT_DIR,
    "DEEPGENOME_API_BASE_URL": API_BASE_URL,
    "DEEPGENOME_API_KEY": API_KEY,
    "DEEPGENOME_JUDGE_MODEL": JUDGE_MODEL,
}
missing_live_configuration = [
    name for name, value in required_live_configuration.items() if not value
]
live_results: list[dict[str, object]] = []
if not RUN_LIVE_JUDGING:
    print("SKIPPED: Live judging is disabled; offline reproducibility checks completed.")
elif missing_live_configuration:
    print(
        "SKIPPED: Live judging configuration is incomplete: "
        + ", ".join(missing_live_configuration)
    )
else:
    live_results = await run_live_evaluation()
    print(f"Completed {len(live_results)} live gene-model evaluations.")

model_log_summaries = aggregate_configured_model_logs(
    MODEL_IDS,
    gene_ids,
    JUDGMENT_DIR,
    threshold=GENE_HIGH_CONTRADICTION_THRESHOLD,
)
if JUDGMENT_DIR is None:
    print("SKIPPED: DEEPGENOME_JUDGMENT_DIR is unset; no private logs were aggregated.")
elif not JUDGMENT_DIR.is_dir():
    print(f"SKIPPED: Judgment directory does not exist: {JUDGMENT_DIR}")
else:
    inventory = {
        model_id: {
            "valid": summary["valid_log_count"],
            "missing": summary["missing_log_count"],
            "invalid": summary["invalid_log_count"],
        }
        for model_id, summary in model_log_summaries.items()
    }
    print(json.dumps(inventory, indent=2, sort_keys=True))
    if not any(
        summary["valid_log_count"]
        for summary in model_log_summaries.values()
    ):
        print(
            "SKIPPED: No valid judgment logs were available; "
            "no current model scores were calculated."
        )

## Results

The values below are labeled **prior-run reference values**. They document
the historical private run from the exploratory analysis and are not
presented as outputs of this offline execution. When a judgment directory is
configured, freshly aggregated summaries appear separately.

In [ ]:
PRIOR_RUN_REFERENCE_VALUES = {
    "label": "prior-run reference values",
    "high_contradiction_rate_at_0.6": {
        "phytomni": 0.0,
        "gemini": 0.08,
        "openai": 0.2,
        "grok": 0.61,
    },
    "mean_contradiction_ratio": {
        "phytomni": 0.12216996785802783,
        "gemini": 0.43429823819272273,
        "openai": 0.4562070516040867,
        "grok": 0.6445983597035421,
    },
}
print(json.dumps(PRIOR_RUN_REFERENCE_VALUES, indent=2, sort_keys=True))
if model_log_summaries:
    current_results = {
        model_id: {
            "high_contradiction_rate": summary["high_contradiction_rate"],
            "average_contradiction_ratio": summary[
                "average_contradiction_ratio"
            ],
            "valid_log_count": summary["valid_log_count"],
        }
        for model_id, summary in model_log_summaries.items()
        if summary["valid_log_count"]
    }
    if current_results:
        print("Current configured-log results:")
        print(json.dumps(current_results, indent=2, sort_keys=True))